# Reading Zarr to CUDA memory

## Overview

In this tutorial, you learn:

- How to read Zarr stores into an `xarray.DataArray` backed by CuPy arrays

In [1]:
import os

import obstore
import obstore.auth.planetary_computer
import planetary_computer
import pystac_client
import xarray as xr
import zarr
from xarray.coders import CFDatetimeCoder

import cupy_xarray  # registers cupy accessor

## Setup Zarr store connection

We'll use the
[Daymet Monthly Hawaii](https://planetarycomputer.microsoft.com/dataset/daymet-monthly-hi)
dataset as an example.

In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1/",
    modifier=planetary_computer.sign_inplace,
)

In [3]:
# https://planetarycomputer.microsoft.com/dataset/daymet-monthly-hi#Example-Notebook
asset = catalog.get_collection(collection_id="daymet-monthly-hi").assets["zarr-abfs"]

The Zarr store requires authentication, we'll follow
https://developmentseed.org/obstore/v0.11.0/examples/zarr/#example
to set up the proper credentials to Planetary Computer.

In [4]:
credential_provider = (
    obstore.auth.planetary_computer.PlanetaryComputerCredentialProvider.from_asset(
        asset=asset
    )
)
azure_store = obstore.store.from_url(
    url=asset.href, credential_provider=credential_provider
)
zarr_store = zarr.storage.ObjectStore(store=azure_store, read_only=True)

## Reading Zarr, first to CPU, then to GPU

This is a two step process, where we'll load the data into CPU memory first,
followed by a host (CPU) to device (GPU) transfer.

First step can be done using
[`xr.open_zarr`](https://docs.xarray.dev/en/stable/generated/xarray.open_zarr.html)

In [5]:
# Read Zarr store
ds_cpu: xr.Dataset = xr.open_zarr(
    store=zarr_store, chunks={}, decode_times=CFDatetimeCoder(use_cftime=True)
)

To keep things manageable, we'll just extract the 'precipitation' variable for 1 year

In [6]:
da_cpu: xr.DataArray = ds_cpu.prcp.sel(time=slice("2020-01-01", "2020-12-31"))
da_cpu

<xarray.DataArray 'prcp' (time: 12, y: 584, x: 284)> Size: 8MB
dask.array<getitem, shape=(12, 584, 284), dtype=float32, chunksize=(12, 584, 284), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 96B 2020-01-16 12:00:00 ... 2020-12-16 00:00:00
  * y        (y) float32 2kB -3.9e+04 -4e+04 -4.1e+04 ... -6.21e+05 -6.22e+05
  * x        (x) float32 1kB -5.802e+06 -5.801e+06 ... -5.52e+06 -5.519e+06
    lat      (y, x) float32 663kB dask.array<chunksize=(584, 284), meta=np.ndarray>
    lon      (y, x) float32 663kB dask.array<chunksize=(584, 284), meta=np.ndarray>
Attributes:
    cell_methods:  area: mean time: sum within days time: sum over days
    grid_mapping:  lambert_conformal_conic
    long_name:     monthly total precipitation
    units:         mm

Note that no data has been loaded yet, everything is still in a lazy Dask array.

### Transfer to GPU memory

Next, we'll call on [`cupy-xarray`](https://cupy-xarray.readthedocs.io/)'s
accessor method
[`.as_cupy()`](https://cupy-xarray.readthedocs.io/latest/generated/xarray.DataArray.cupy.as_cupy.html)
to convert the underlying array from NumPy (CPU) to CuPy (GPU).

In [7]:
# Lazy conversion of underlying array from NumPy (CPU) to CuPy (GPU)
da_gpu = da_cpu.as_cupy()
da_gpu.data

dask.array<asarray, shape=(12, 584, 284), dtype=float32, chunksize=(12, 584, 284), chunktype=cupy.ndarray>

Again, no data has actually been loaded yet.
We'll need to call
[`.compute()`](https://docs.xarray.dev/en/latest/generated/xarray.DataArray.compute.html)
to trigger loading of the data into GPU memory.

In [8]:
# Materialize data in GPU memory
da_gpu = da_gpu.compute()
print(type(da_gpu.data))
da_gpu

<class 'cupy.ndarray'>


<xarray.DataArray 'prcp' (time: 12, y: 584, x: 284)> Size: 8MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]],
      shape=(12, 584, 284), dtype=float32)
Coordinates:
  * time     (time) object 96B 2020-01-16 12:00:00 ... 2020-12-16 00:00:00
  * y        (y) float32 2kB -3.9e+04 -4e+04 -4.1e+04 ... -6.21e+05 -6.22e+05
  * x        (x) float32 1kB -5.802e+06 -5.801e+06 ... -5.52e+06 -5.519e+06
    lat      (y, x) float32 663kB 21.87 21.87 21.88 21.88 ... 19.48 19.48 19.49
    lon      (y, x) float32 663kB -160.3 -160.3 -160.3 ... -154.8 -154.8 -154.8
Attributes:
    cell_methods:  area: mean time: sum within days time: sum over days
    grid_mapping:  lambert_conformal_conic
    long_name:     monthly total precipitation
    units:         mm

In the next section, we'll show how to read Zarr to GPU using an alternative method.
It requires having the Zarr v3 formatted store locally, so let's save it here.

In [9]:
# TypeError converting v2 to v3 - https://github.com/zarr-developers/zarr-python/issues/2964
for coord in ds_cpu.coords:
    ds_cpu[coord].encoding["compressors"] = (zarr.codecs.ZstdCodec(),)
ds_cpu.prcp.encoding["compressors"] = (zarr.codecs.ZstdCodec(),)
ds_cpu.prcp.to_zarr(
    store="/tmp/daymet_2020.zarr", mode="w", zarr_format=3, consolidated=False
)

## Alternative way of reading

This is a more advanced way of reading data from Zarr stores to the GPU.
You will need:
- To [install `kvikio`](https://docs.rapids.ai/api/kvikio/stable/install/)
- The Zarr store to be on a local drive
- (ideally) a system with NVIDIA GPUDirectStorage (GDS) configured,
  though if not, there is a fallback mode

References:
- https://xarray.dev/blog/gpu-pipeline#step-2-direct-to-gpu-data-reading-with-zarr-python-3--kvikio-
- https://zarr.readthedocs.io/en/v3.2.1/user-guide/gpu/#using-gpus-with-zarr

In [10]:
import kvikio.zarr

In [11]:
store = kvikio.zarr.GDSStore(root="/tmp/daymet_2020.zarr")

with zarr.config.enable_gpu():
    ds_gpu_direct = xr.open_zarr(store=store, consolidated=False, chunks=None)
    ds_gpu_direct.load()  # ensure data is loaded to GPU inside with-block
print(type(ds_gpu_direct.prcp.data))
ds_gpu_direct.prcp

<class 'cupy.ndarray'>


<xarray.DataArray 'prcp' (time: 492, y: 584, x: 284)> Size: 326MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]],
      shape=(492, 584, 284), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 4kB 1980-01-16T12:00:00 ... 2020-12-16
  * y        (y) float32 2kB -3.9e+04 -4e+04 -4.1e+04 ... -6.21e+05 -6.22e+05
  * x        (x) float32 1kB -5.802e+06 -5.801e+06 ... -5.52e+06 -5.519e+06
    lat      (y, x) float32 663kB array([[21.865978, 21.871851, 21.877724, .....
    lon      (y, x) float32 663kB array([[-160.29884, -160.2917 , -160.28458,...
Attributes:
    cell_methods:  area: mean time: sum within days time: sum over days
    grid_mapping:  lambert_conformal_conic
    long_name:     monthly total precipitation
    units:         mm

In the future, the direct to GPU loading interface could be more ergonomic
after the work done in https://github.com/xarray-contrib/cupy-xarray/pull/70
is completed.